# 02a — Tier 1 Bake-Off: Gemma 4 E2B vs Phi-4-mini-instruct

**Runs on:** Google Colab Free (T4 15 GB) — no other setup needed.

## How to use this notebook
1. Open in Colab. Set runtime → GPU (T4).
2. **Runtime → Run all.**
3. When prompted, upload `train.jsonl`, `val.jsonl`, and any `test_*.jsonl` files (from `tools/yen_sei/data/refined/` on your local machine).
4. Wait ~30–45 min. The notebook will:
   - QLoRA fine-tune both candidates on a small slice (100 train / 20 val, 1 epoch each).
   - Compare format compliance, output length, train time, and peak VRAM.
   - Print a verdict and save `winner.json` to `/content/`.
5. Note the winner. Use it as `MODEL_NAME` in `02_train_tier1.ipynb`.

**Total user effort: pick the runtime, click Run all, drop the data files. Nothing to copy-paste.**

In [ ]:
# =====================================================================
# Cell 1 — Install dependencies (silent). ~2-3 min on first run.
# =====================================================================
import subprocess, sys, os, time

PKGS = [
    "unsloth",
    "peft>=0.11.0",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
    "trl>=0.10.0",
]
print("Installing dependencies (silent)…")
t0 = time.time()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
    check=True,
)
print(f"Done in {time.time() - t0:.1f}s.")


In [ ]:
# =====================================================================
# Cell 1.5 - Bundle tools.yen_sei.eval into Colab.
#
# The notebook imports `from tools.yen_sei.eval.runner import evaluate_test_sets`
# but Colab does not have the yen-go repo. This cell writes the 4 small
# files (~22 KB total) that the eval runner needs into /content/yen_sei_bundle/
# and adds it to sys.path. No repo clone, no extra upload required.
#
# To regenerate after editing runner/scorers/judges:
#   python tools/yen_sei/notebooks/_gen_bundle_cell.py
# then paste the output back into both notebooks.
# =====================================================================
import sys
from pathlib import Path

BUNDLE_ROOT = Path("/content/yen_sei_bundle")

_GO_CONST = r'''"""Colab stub: only GO_TECHNIQUE_PATTERN is used by scorers.py."""
import re

GO_TECHNIQUES = frozenset({
    "net", "geta", "ladder", "shicho", "snapback", "ko", "seki",
    "tesuji", "life", "death", "kill", "capture", "connect", "cut",
    "eye", "liberty", "liberties", "atari", "shortage", "semeai",
    "throw-in", "squeeze", "placement", "hane", "descent", "clamp",
    "wedge", "peep", "bamboo", "tiger", "empty triangle", "ponnuki",
    "alive", "dead", "unconditionally", "approach", "invasion",
    "shape", "joseki", "proverb", "sacrifice", "damezumari",
    "nakade", "bent four", "bulky five", "rabbity six",
    "false eye", "double atari", "under the stones",
})

GO_TECHNIQUE_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in sorted(GO_TECHNIQUES, key=len, reverse=True)) + r")\b",
    re.IGNORECASE,
)
'''

_TEACHING_SCHEMA = r'''"""Colab stub: tagged text format parser for v3 SFT output."""
import re

_SECTION_RE = re.compile(r"^---(CORRECT|WRONG|HINT)---$", re.MULTILINE)


def parse_tagged_text(text: str) -> tuple[str, list[str], list[str]]:
    """Parse tagged text -> (correct_comment, wrong_comments, hints).

    Raises ValueError if no ---CORRECT--- section found.
    """
    tokens = _SECTION_RE.split(text)
    correct_comment = ""
    wrong_comments: list[str] = []
    hints: list[str] = []
    i = 0
    while i < len(tokens):
        token = tokens[i].strip()
        if token == "CORRECT" and i + 1 < len(tokens):
            correct_comment = tokens[i + 1].strip()
            i += 2
        elif token == "WRONG" and i + 1 < len(tokens):
            content = tokens[i + 1].strip()
            if content:
                wrong_comments.append(content)
            i += 2
        elif token == "HINT" and i + 1 < len(tokens):
            content = tokens[i + 1].strip()
            if content:
                hints.append(content)
            i += 2
        else:
            i += 1
    if not correct_comment:
        raise ValueError("No ---CORRECT--- section found in tagged text")
    return correct_comment, wrong_comments, hints
'''

_SCORERS = r'''"""Layer A (structural) and Layer B (grounded) scorers. Pure, deterministic, free.

Supports both tagged text (v3+) and JSON (backward compat) output formats.
"""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass

from tools.core.go_teaching_constants import GO_TECHNIQUE_PATTERN
from tools.core.teaching_schema import parse_tagged_text

_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)
_SGF_COORD_RE = re.compile(r"\b[a-s]{2}\b")


@dataclass
class StructuralScore:
    parsed_ok: bool
    has_correct: bool
    has_wrong: bool
    n_hints: int
    n_chars_correct: int
    n_chars_wrong: int


@dataclass
class GroundedScore:
    mentions_correct_move: bool
    mentions_any_tag_technique: bool
    no_off_board_coords: bool
    looks_english: bool
    techniques_matched: list[str]
    correct_move_coord: str


def _score_structural_tagged(generated: str) -> StructuralScore:
    """Score tagged text format (---CORRECT---/---WRONG---/---HINT---)."""
    try:
        correct, wrongs, hints = parse_tagged_text(generated)
    except ValueError:
        return StructuralScore(False, False, False, 0, 0, 0)

    wc_text = " ".join(wrongs)
    return StructuralScore(
        parsed_ok=True,
        has_correct=bool(correct.strip()),
        has_wrong=bool(wc_text.strip()),
        n_hints=len(hints),
        n_chars_correct=len(correct),
        n_chars_wrong=len(wc_text),
    )


def _score_structural_json(generated: str) -> StructuralScore | None:
    """Try to score as JSON format. Returns None if not JSON."""
    blob = _JSON_BLOCK_RE.search(generated or "")
    if not blob:
        return None
    try:
        obj = json.loads(blob.group(0))
    except json.JSONDecodeError:
        return None

    tc = obj.get("teaching_comments") or {}
    cc = (tc.get("correct_comment") or "").strip() if isinstance(tc, dict) else ""
    wc = tc.get("wrong_comments") if isinstance(tc, dict) else None
    if isinstance(wc, dict):
        wc_text = " ".join(str(v) for v in wc.values())
    elif isinstance(wc, list):
        wc_text = " ".join(str(v) for v in wc)
    else:
        wc_text = ""
    hints = obj.get("hints") or []
    return StructuralScore(
        parsed_ok=True,
        has_correct=bool(cc),
        has_wrong=bool(wc_text.strip()),
        n_hints=len(hints) if isinstance(hints, list) else 0,
        n_chars_correct=len(cc),
        n_chars_wrong=len(wc_text),
    )


def score_structural(generated: str) -> StructuralScore:
    """Layer A: did the model emit a parseable response with the right shape?

    Supports both JSON (backward compat) and tagged text (v3+) formats.
    Tries JSON first; falls back to tagged text.
    """
    json_result = _score_structural_json(generated)
    if json_result is not None:
        return json_result
    return _score_structural_tagged(generated)


def _flatten_text(parsed_obj: dict) -> str:
    out: list[str] = []

    def walk(x):
        if isinstance(x, str):
            out.append(x)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(parsed_obj)
    return " ".join(out)


def score_grounded(
    generated: str,
    correct_move_coord: str,
    puzzle_tags: list[str],
    setup_coords: list[str],
) -> GroundedScore:
    blob = _JSON_BLOCK_RE.search(generated or "")
    parsed: dict = {}
    if blob:
        try:
            parsed = json.loads(blob.group(0))
        except json.JSONDecodeError:
            parsed = {}
    text = _flatten_text(parsed) if parsed else (generated or "")
    text_lower = text.lower()

    mentions_correct = bool(correct_move_coord) and (correct_move_coord.lower() in text_lower)

    techniques_matched: list[str] = []
    tag_tokens: set[str] = set()
    for t in puzzle_tags:
        for piece in re.split(r"[-_,\s]+", t.lower()):
            if piece:
                tag_tokens.add(piece)
    for m in GO_TECHNIQUE_PATTERN.finditer(text_lower):
        word = m.group(1).lower()
        if word in tag_tokens and word not in techniques_matched:
            techniques_matched.append(word)
    mentions_tag_tech = bool(techniques_matched) or not tag_tokens

    coord_set = {c.lower() for c in setup_coords if isinstance(c, str)}
    if correct_move_coord:
        coord_set.add(correct_move_coord.lower())
    found_coords = _SGF_COORD_RE.findall(text_lower)
    unknown = [c for c in found_coords if c not in coord_set]
    no_off_board = (len(unknown) <= max(1, len(found_coords) // 3))

    letters = sum(1 for ch in text if "a" <= ch.lower() <= "z")
    looks_english = (letters / max(len(text), 1)) >= 0.6 if text else False

    return GroundedScore(
        mentions_correct_move=mentions_correct,
        mentions_any_tag_technique=mentions_tag_tech,
        no_off_board_coords=no_off_board,
        looks_english=looks_english,
        techniques_matched=techniques_matched,
        correct_move_coord=correct_move_coord,
    )


def to_dict(obj) -> dict:
    return asdict(obj)
'''

_JUDGES = r'''"""Layer C judges. Notebook depends only on Judge protocol; pick a backend at runtime."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Protocol


@dataclass
class JudgeResult:
    score: float
    rubric: dict
    rationale: str = ""
    backend: str = ""
    raw: dict = field(default_factory=dict)


class Judge(Protocol):
    def grade(
        self,
        prompt: str,
        generated: str,
        reference: str | None,
        metadata: dict,
    ) -> JudgeResult: ...

    def finalize(self) -> None: ...


class ManualJudge:
    def __init__(self, queue_path: Path | str, sample_n: int = 20):
        self.queue_path = Path(queue_path)
        self.sample_n = sample_n
        self._buffer: list[dict] = []

    def grade(
        self,
        prompt: str,
        generated: str,
        reference: str | None,
        metadata: dict,
    ) -> JudgeResult:
        self._buffer.append({
            "prompt": prompt,
            "generated": generated,
            "reference": reference,
            "metadata": metadata,
            "score": None,
            "rationale": "",
        })
        return JudgeResult(score=-1.0, rubric={}, rationale="awaiting human review", backend="manual")

    def finalize(self) -> None:
        self.queue_path.parent.mkdir(parents=True, exist_ok=True)
        items = self._buffer
        if len(items) > self.sample_n:
            import random
            random.seed(42)
            items = random.sample(items, self.sample_n)
        with self.queue_path.open("w", encoding="utf-8") as f:
            for it in items:
                f.write(json.dumps(it, ensure_ascii=False) + "\n")


def to_dict(r: JudgeResult) -> dict:
    return asdict(r)
'''

_RUNNER = r'''"""Inference + scoring loop. Notebook entry point: evaluate(...)."""
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Optional

from .judges import Judge
from .scorers import GroundedScore, StructuralScore, score_grounded, score_structural, to_dict

_USER_BOARD_RE = re.compile(r"Board: (\d+)x\d+")
_USER_TAGS_RE = re.compile(r"^Tags: (.+)$", re.MULTILINE)
_USER_BLACK_RE = re.compile(r"^Black stones: (.+)$", re.MULTILINE)
_USER_WHITE_RE = re.compile(r"^White stones: (.+)$", re.MULTILINE)


def _parse_user_prompt(text: str) -> tuple[list[str], list[str], list[str]]:
    tags: list[str] = []
    black: list[str] = []
    white: list[str] = []
    if (m := _USER_TAGS_RE.search(text)):
        tags = [t.strip() for t in m.group(1).split(",") if t.strip()]
    if (m := _USER_BLACK_RE.search(text)):
        black = [c.strip() for c in m.group(1).split(",") if c.strip()]
    if (m := _USER_WHITE_RE.search(text)):
        white = [c.strip() for c in m.group(1).split(",") if c.strip()]
    return tags, black, white


def _correct_move_from_reference(reference_assistant: str | None) -> str:
    if not reference_assistant:
        return ""
    m = re.search(r"\{!([a-s]{2})\}", reference_assistant)
    return m.group(1) if m else ""


def generate_one(model, tokenizer, messages: list[dict], max_new_tokens: int = 384) -> str:
    import torch
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def evaluate(
    model,
    tokenizer,
    test_rows: list[dict],
    out_dir: str | Path,
    judge: Optional[Judge] = None,
    max_new_tokens: int = 384,
    log_every: int = 25,
    sidecars: list[dict] | None = None,
    test_set_id: str = "",
) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n = len(test_rows)
    results: list[dict] = []
    t0 = time.time()

    print(f"[eval] running on {n} test rows")
    for i, row in enumerate(test_rows):
        msgs = row["messages"]
        prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
        reference = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        user_text = next((m["content"] for m in msgs if m["role"] == "user"), "")

        generated = generate_one(model, tokenizer, prompt_msgs, max_new_tokens=max_new_tokens)

        struct = score_structural(generated)

        tags, black, white = _parse_user_prompt(user_text)
        if sidecars is not None and i < len(sidecars):
            sc = sidecars[i] or {}
            correct_move = sc.get("correct_first_move") or _correct_move_from_reference(reference)
            sc_tags = sc.get("tags") or sc.get("techniques_found") or []
            if sc_tags and not tags:
                tags = list(sc_tags)
        else:
            correct_move = _correct_move_from_reference(reference)
        ground = score_grounded(
            generated,
            correct_move_coord=correct_move,
            puzzle_tags=tags,
            setup_coords=black + white,
        )

        judge_result = None
        if judge is not None:
            jr = judge.grade(
                prompt=user_text,
                generated=generated,
                reference=reference,
                metadata={"tags": tags, "correct_move_coord": correct_move},
            )
            judge_result = to_dict(jr)

        results.append({
            "idx": i,
            "structural": to_dict(struct),
            "grounded": to_dict(ground),
            "judge": judge_result,
            "generated_preview": generated[:600],
            "reference_preview": (reference or "")[:600],
        })

        if (i + 1) % log_every == 0:
            print(f"  {i + 1}/{n} ({(time.time() - t0) / 60:.1f} min elapsed)")

    if judge is not None:
        judge.finalize()

    elapsed_min = (time.time() - t0) / 60
    summary = _aggregate(results)
    summary["n"] = n
    summary["elapsed_minutes"] = round(elapsed_min, 2)

    (out_dir / "test_results.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (out_dir / "test_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    _print_summary(summary)
    return summary


def _aggregate(results: list[dict]) -> dict:
    n = max(len(results), 1)
    s = [r["structural"] for r in results]
    g = [r["grounded"] for r in results]

    pct = lambda flag_list: round(100.0 * sum(1 for x in flag_list if x) / n, 1)

    summary = {
        "structural": {
            "format_compliance_pct": pct([x["parsed_ok"] for x in s]),
            "has_correct_pct": pct([x["has_correct"] for x in s]),
            "has_wrong_pct": pct([x["has_wrong"] for x in s]),
            "avg_hints": round(sum(x["n_hints"] for x in s) / n, 2),
            "avg_chars_correct": round(sum(x["n_chars_correct"] for x in s) / n, 1),
            "avg_chars_wrong": round(sum(x["n_chars_wrong"] for x in s) / n, 1),
        },
        "grounded": {
            "mentions_correct_move_pct": pct([x["mentions_correct_move"] for x in g]),
            "mentions_tag_technique_pct": pct([x["mentions_any_tag_technique"] for x in g]),
            "no_off_board_coords_pct": pct([x["no_off_board_coords"] for x in g]),
            "looks_english_pct": pct([x["looks_english"] for x in g]),
        },
    }

    useful = sum(
        1 for r in results
        if r["structural"]["parsed_ok"]
        and r["structural"]["has_correct"]
        and (r["grounded"]["mentions_correct_move"] or r["grounded"]["mentions_any_tag_technique"])
        and r["grounded"]["looks_english"]
    )
    summary["useful_answer_pct"] = round(100.0 * useful / n, 1)
    return summary


def evaluate_test_sets(
    model,
    tokenizer,
    test_sets: dict,
    out_dir: str | Path,
    judge: Optional[Judge] = None,
    max_new_tokens: int = 384,
    log_every: int = 25,
) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    per_set: dict[str, dict] = {}
    for ts_id, bundle in test_sets.items():
        chat_rows = bundle.get("chat_rows") or []
        sidecars = bundle.get("sidecars")
        if not chat_rows:
            print(f"[eval] skipping {ts_id}: no rows")
            continue
        print(f"\n[eval] === test set: {ts_id} ===")
        per_set[ts_id] = evaluate(
            model, tokenizer, chat_rows, out_dir / ts_id,
            judge=judge, max_new_tokens=max_new_tokens, log_every=log_every,
            sidecars=sidecars, test_set_id=ts_id,
        )

    cmp = {
        ts_id: {
            "n": s.get("n"),
            "useful_answer_pct": s.get("useful_answer_pct"),
            "format_compliance_pct": s.get("structural", {}).get("format_compliance_pct"),
            "mentions_correct_move_pct": s.get("grounded", {}).get("mentions_correct_move_pct"),
            "mentions_tag_technique_pct": s.get("grounded", {}).get("mentions_tag_technique_pct"),
            "no_off_board_coords_pct": s.get("grounded", {}).get("no_off_board_coords_pct"),
            "looks_english_pct": s.get("grounded", {}).get("looks_english_pct"),
        }
        for ts_id, s in per_set.items()
    }
    (out_dir / "comparison.json").write_text(
        json.dumps(cmp, indent=2), encoding="utf-8",
    )
    print("\n=== TEST-SET COMPARISON ===")
    cols = ["n", "useful_answer_pct", "format_compliance_pct",
            "mentions_correct_move_pct", "mentions_tag_technique_pct",
            "no_off_board_coords_pct", "looks_english_pct"]
    header = f"{'test_set':<32}" + "".join(f"{c:>12}" for c in cols)
    print(header)
    print("-" * len(header))
    for ts_id, row in cmp.items():
        line = f"{ts_id:<32}" + "".join(f"{str(row.get(c, '')):>12}" for c in cols)
        print(line)
    return {"per_set": per_set, "comparison": cmp}


def load_test_set_bundle(refined_dir: str | Path, test_set_id: str) -> dict:
    refined_dir = Path(refined_dir)
    chat_path = refined_dir / f"test_{test_set_id}.jsonl"
    meta_path = refined_dir / f"test_{test_set_id}_metadata.jsonl"
    chat_rows: list[dict] = []
    sidecars: list[dict] = []
    if chat_path.exists():
        with chat_path.open("r", encoding="utf-8") as f:
            for line in f:
                chat_rows.append(json.loads(line))
    if meta_path.exists():
        with meta_path.open("r", encoding="utf-8") as f:
            for line in f:
                sidecars.append(json.loads(line))
    return {"chat_rows": chat_rows, "sidecars": sidecars or None}


def _print_summary(summary: dict) -> None:
    print("\n=== EVAL SUMMARY ===")
    print(f"  test rows:         {summary.get('n')}")
    print(f"  elapsed:           {summary.get('elapsed_minutes')} min")
    print(f"  USEFUL ANSWER %:   {summary['useful_answer_pct']}    <-- single headline metric")
    print("  -- Structural (Layer A) --")
    for k, v in summary["structural"].items():
        print(f"    {k:<24} {v}")
    print("  -- Grounded (Layer B) --")
    for k, v in summary["grounded"].items():
        print(f"    {k:<28} {v}")
'''

_FILES = {
    "tools/__init__.py": "",
    "tools/core/__init__.py": "",
    "tools/core/go_teaching_constants.py": _GO_CONST,
    "tools/core/teaching_schema.py": _TEACHING_SCHEMA,
    "tools/yen_sei/__init__.py": "",
    "tools/yen_sei/eval/__init__.py": "",
    "tools/yen_sei/eval/scorers.py": _SCORERS,
    "tools/yen_sei/eval/judges.py": _JUDGES,
    "tools/yen_sei/eval/runner.py": _RUNNER,
}
for _rel, _content in _FILES.items():
    _p = BUNDLE_ROOT / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_content, encoding="utf-8")

if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

from tools.yen_sei.eval.runner import evaluate_test_sets  # noqa: F401 sanity check
print(f"yen_sei eval bundle ready at {BUNDLE_ROOT}")

In [ ]:
# =====================================================================
# Cell 2 — CONFIG. Edit only if you want different models or sizes.
# =====================================================================

CANDIDATES = {
    # Use the Unsloth 4-bit checkpoint for T4 stability (still Gemma 4 E2B).
    "gemma":  "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    "phi4":   "microsoft/Phi-4-mini-instruct",
}

# Bake-off uses small slice for fast comparison; full SFT is in 02_train_tier1.
# T4-safe defaults: keep effective batch size (= 16) while lowering per-step VRAM.
TRAIN_LIMIT = 100        # examples for QLoRA fine-tune
EVAL_LIMIT  = 20         # held-out examples for inference comparison
MAX_SEQ_LEN = 768        # 1024 can OOM on T4 after repeated runs in one session
EPOCHS      = 1
BATCH_SIZE  = 2
GRAD_ACCUM  = 8
LR          = 2e-4
LORA_R      = 16
LORA_ALPHA  = 32

OUT_DIR = "/content/yen_sei_bakeoff"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Candidates: {list(CANDIDATES.values())}")
print(f"Output dir: {OUT_DIR}")


## Upload data

The next cell looks for the data files in `/content/` (Colab session storage). If they are missing, it opens an **upload widget** — just select all the files at once from `tools/yen_sei/data/refined/` on your local machine. **No Drive mount needed.**

Files to upload:

| File | Required? | Purpose |
|------|-----------|---------|
| `train.jsonl` | **Yes** | QLoRA fine-tune examples |
| `val.jsonl` | **Yes** | Held-out rows for reference |
| `test_marker_only_*.jsonl` | Recommended | Marker-only test sets (used for winner decision) |
| `test_*_metadata.jsonl` | Optional | Sidecar files with correct-move coords + tags |

After uploading, files stay in `/content/` for the entire Colab session. If the session restarts, re-run this cell and upload again (or mount Drive for persistence).


In [ ]:
# =====================================================================
# Cell 3 — Locate train.jsonl + val.jsonl + named TEST SETS.
#
# Files are looked up in /content/ first (Colab session storage).
# If train.jsonl or val.jsonl are missing, an upload widget opens —
# select ALL files at once (train, val, test_*.jsonl, metadata sidecars).
# No Drive mount is needed; uploaded files persist for the session.
# =====================================================================
import json
from pathlib import Path

REQUIRED = ["train.jsonl", "val.jsonl"]
SEARCH_DIRS = [Path("/content"), Path.cwd(), Path("/content/drive/MyDrive/yen_sei")]

def find_data():
    found = {}
    for name in REQUIRED:
        for d in SEARCH_DIRS:
            p = d / name
            if p.exists():
                found[name] = p
                break
    return found

found = find_data()
missing = [n for n in REQUIRED if n not in found]

if missing:
    print(f"Missing required files: {missing}")
    print("Opening upload widget — select train.jsonl, val.jsonl, and any test_*.jsonl files at once.")
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = Path("/content") / name
            dest.write_bytes(data)
            print(f"  saved {name} -> {dest} ({len(data):,} bytes)")
        found = find_data()
    except ImportError:
        raise SystemExit(
            "Not in Colab. Place train.jsonl, val.jsonl + test_*.jsonl next to "
            "this notebook or in /content/."
        )

assert all(n in found for n in REQUIRED), f"Still missing: {set(REQUIRED) - set(found)}"
TRAIN_PATH = found["train.jsonl"]
VAL_PATH = found["val.jsonl"]
DATA_DIR = TRAIN_PATH.parent
print(f"\ntrain: {TRAIN_PATH}\nval:   {VAL_PATH}")

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

train_rows = load_jsonl(TRAIN_PATH, limit=TRAIN_LIMIT)
val_rows = load_jsonl(VAL_PATH, limit=EVAL_LIMIT)
print(f"\nLoaded {len(train_rows)} train rows, {len(val_rows)} val rows.")
print(f"First example messages roles: {[m['role'] for m in train_rows[0]['messages']]}")

# Discover all test_*.jsonl in DATA_DIR (excluding *_metadata.jsonl).
# Searches both /content/ and DATA_DIR so uploaded test files are found
# even if train.jsonl was already present in a different search dir.
test_search_dirs = list({DATA_DIR, Path("/content")})
test_set_files = sorted(
    p
    for d in test_search_dirs
    for p in d.glob("test_*.jsonl")
    if not p.name.endswith("_metadata.jsonl")
)
TEST_SETS: dict[str, dict] = {}
for chat_path in test_set_files:
    ts_id = chat_path.stem.removeprefix("test_")
    # Sidecar may live alongside the chat file or in /content/
    meta_path = next(
        (d / f"test_{ts_id}_metadata.jsonl" for d in test_search_dirs
         if (d / f"test_{ts_id}_metadata.jsonl").exists()),
        None,
    )
    chat_rows = load_jsonl(chat_path, limit=EVAL_LIMIT)
    sidecars = load_jsonl(meta_path, limit=EVAL_LIMIT) if meta_path else None
    TEST_SETS[ts_id] = {"chat_rows": chat_rows, "sidecars": sidecars}
    print(f"  test set '{ts_id}': {len(chat_rows)} rows"
          f"{' (with sidecar)' if sidecars else ''}")

if not TEST_SETS:
    print("\nWARNING: no test_*.jsonl found. Falling back to val_rows.")
    TEST_SETS = {"val_fallback": {"chat_rows": val_rows, "sidecars": None}}


In [ ]:
# =====================================================================
# Cell 4 - Helpers: train one candidate, evaluate against the named
# test sets via tools.yen_sei.eval.runner.
# =====================================================================
import gc, json, time, torch
from pathlib import Path
import tools.yen_sei.eval.runner as eval_runner


def _vram_peak_gb():
    return torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0


def _reset_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        torch.cuda.reset_peak_memory_stats()


def _purge_stale_model_refs():
    """Delete common notebook globals that can keep CUDA memory alive."""
    g = globals()
    stale_names = ["m", "model", "tok", "tokenizer", "trainer", "peft_model"]
    removed = []
    for name in stale_names:
        if name in g:
            try:
                del g[name]
                removed.append(name)
            except Exception:
                pass
    if removed:
        print(f"  released stale globals: {', '.join(removed)}")


def _cuda_mem_line():
    if not torch.cuda.is_available():
        return "cuda unavailable"
    free_b, total_b = torch.cuda.mem_get_info()
    return f"free={free_b / (1024 ** 3):.2f} GB / total={total_b / (1024 ** 3):.2f} GB"


def _seq_len_attempts():
    # Progressively reduce seq length to fit Gemma 4 on T4 if memory is tight.
    candidates = [int(MAX_SEQ_LEN), 768, 640, 512]
    out = []
    for s in candidates:
        if s > 0 and s not in out:
            out.append(s)
    return out


def _load_model_with_t4_retry(model_id):
    """Load 4-bit model with retries for CPU/disk dispatch and OOM on Colab T4."""
    from unsloth import FastLanguageModel  # type: ignore

    _purge_stale_model_refs()
    _reset_vram()

    attempts = []
    for seq_len in _seq_len_attempts():
        base = {
            "model_name": model_id,
            "max_seq_length": seq_len,
            "load_in_4bit": True,
            "dtype": None,
        }
        attempts.append({
            "label": f"{model_id} (seq={seq_len}, auto-map)",
            "kwargs": dict(base),
        })
        attempts.append({
            "label": f"{model_id} (seq={seq_len}, gpu-only)",
            "kwargs": {
                **base,
                "device_map": {"": 0},
                "gpu_memory_utilization": 0.92,
            },
        })

    last_err = None
    for i, attempt in enumerate(attempts, start=1):
        try:
            print(f"  load attempt {i}/{len(attempts)}: {attempt['label']} | {_cuda_mem_line()}")
            model, tokenizer = FastLanguageModel.from_pretrained(**attempt["kwargs"])
            return model, tokenizer
        except Exception as e:
            last_err = e
            msg = str(e).splitlines()[0]
            print(f"  load attempt failed: {type(e).__name__}: {msg}")
            _reset_vram()

    raise RuntimeError(
        "Failed to load model on this GPU after retries. "
        f"Final CUDA state: {_cuda_mem_line()}. "
        "Restart runtime and run from Cell 1, or use a smaller model for this session."
    ) from last_err


def _decode_with_fallback(tokenizer, token_ids):
    if hasattr(tokenizer, "decode"):
        return tokenizer.decode(token_ids, skip_special_tokens=True)
    if hasattr(tokenizer, "tokenizer") and hasattr(tokenizer.tokenizer, "decode"):
        return tokenizer.tokenizer.decode(token_ids, skip_special_tokens=True)
    raise TypeError("Tokenizer/processor does not expose decode().")


def _eos_id_with_fallback(model, tokenizer):
    eos_id = getattr(tokenizer, "eos_token_id", None)
    if eos_id is None and hasattr(tokenizer, "tokenizer"):
        eos_id = getattr(tokenizer.tokenizer, "eos_token_id", None)
    if eos_id is None:
        eos_id = getattr(model.config, "eos_token_id", None)
    if isinstance(eos_id, list):
        eos_id = eos_id[0]
    return eos_id


def _tokenize_prompt_compat(tokenizer, prompt):
    # Gemma 4 processor needs text=..., while plain tokenizers accept positional text.
    try:
        return tokenizer(text=prompt, return_tensors="pt")
    except TypeError:
        return tokenizer(prompt, return_tensors="pt")


def _generate_one_compat(model, tokenizer, messages: list[dict], max_new_tokens: int = 384) -> str:
    import torch

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _tokenize_prompt_compat(tokenizer, prompt)
    inputs = inputs.to(model.device)

    pad_id = _eos_id_with_fallback(model, tokenizer)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    return _decode_with_fallback(tokenizer, out[0][prompt_len:])


# Monkey patch runner generation for processor-based models like Gemma 4.
eval_runner.generate_one = _generate_one_compat
evaluate_test_sets = eval_runner.evaluate_test_sets


def train_one(model_id, train_rows, out_subdir):
    from unsloth import FastLanguageModel  # type: ignore
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig

    _purge_stale_model_refs()
    _reset_vram()
    print(f"\n=== {model_id} ===")
    print(f"  before load: {_cuda_mem_line()}")
    t0 = time.time()

    model, tokenizer = _load_model_with_t4_retry(model_id)

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules="all-linear",
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    def _format(example):
        return {
            "text": tokenizer.apply_chat_template(
                example["messages"], tokenize=False, add_generation_prompt=False
            )
        }

    ds = Dataset.from_list(train_rows).map(_format, remove_columns=["messages"])
    out_dir = os.path.join(OUT_DIR, out_subdir)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        args=SFTConfig(
            output_dir=out_dir,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR,
            lr_scheduler_type="cosine",
            warmup_steps=1,
            weight_decay=0.01,
            logging_steps=10,
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            report_to="none",
            save_strategy="no",
        ),
    )
    trainer.train()
    train_secs = time.time() - t0
    peak = _vram_peak_gb()
    print(f"  train: {train_secs:.0f}s, peak VRAM: {peak:.1f} GB")
    return model, tokenizer, train_secs, peak


def evaluate_candidate(model, tokenizer, test_sets, out_subdir):
    """Run the multi-test-set runner and return its dict output."""
    from unsloth import FastLanguageModel  # type: ignore

    FastLanguageModel.for_inference(model)
    return evaluate_test_sets(
        model=model,
        tokenizer=tokenizer,
        test_sets=test_sets,
        out_dir=Path(OUT_DIR) / out_subdir / "eval",
        judge=None,
        max_new_tokens=384,
        log_every=25,
    )


def free_model(model):
    del model
    _reset_vram()


print("Helpers ready (Layer A+B via tools.yen_sei.eval.runner; Layer C optional).")


## Train + evaluate Candidate A (Gemma)


In [ ]:
# =====================================================================
# Cell 5 — Bake-off Candidate A.
# =====================================================================
results = {}

m, tok, train_secs_a, peak_a = train_one(CANDIDATES["gemma"], train_rows, "gemma")
eval_a = evaluate_candidate(m, tok, TEST_SETS, "gemma")
results["gemma"] = {
    "model": CANDIDATES["gemma"],
    "train_seconds": train_secs_a,
    "peak_vram_gb": peak_a,
    "comparison": eval_a["comparison"],
}
print(json.dumps(results["gemma"], indent=2))

free_model(m)


In [ ]:
# =====================================================================
# Cell 6 — Bake-off Candidate B.
# =====================================================================
m, tok, train_secs_b, peak_b = train_one(CANDIDATES["phi4"], train_rows, "phi4")
eval_b = evaluate_candidate(m, tok, TEST_SETS, "phi4")
results["phi4"] = {
    "model": CANDIDATES["phi4"],
    "train_seconds": train_secs_b,
    "peak_vram_gb": peak_b,
    "comparison": eval_b["comparison"],
}
print(json.dumps(results["phi4"], indent=2))

free_model(m)


## Verdict


In [ ]:
# =====================================================================
# Cell 7 — Compare candidates per test set and pick winner.
#
# Decision rule: highest MEAN useful_answer_pct across the marker-only
# test sets (the prose-leak-free pools). Tie-break by smaller model.
# heldout_gold_silver is reported but excluded from the decision because
# it shares distribution with the training pool.
# =====================================================================
def _mean(xs):
    xs = [x for x in xs if isinstance(x, (int, float))]
    return round(sum(xs) / max(len(xs), 1), 1)

a, b = results["gemma"], results["phi4"]
ts_ids = sorted(set(a["comparison"].keys()) | set(b["comparison"].keys()))

print(f"\nPer-test-set useful_answer_pct:")
print(f"{'test_set':<32} {'gemma':>10} {'phi4':>10}")
print("-" * 54)
for ts in ts_ids:
    av = a["comparison"].get(ts, {}).get("useful_answer_pct", "-")
    bv = b["comparison"].get(ts, {}).get("useful_answer_pct", "-")
    marker = "  *" if ts.startswith("marker_only_") else "   "
    print(f"{ts:<32} {str(av):>10} {str(bv):>10}{marker}")
print("(* = used in winner decision)")

marker_sets = [ts for ts in ts_ids if ts.startswith("marker_only_")]
mean_a = _mean([a["comparison"].get(ts, {}).get("useful_answer_pct") for ts in marker_sets])
mean_b = _mean([b["comparison"].get(ts, {}).get("useful_answer_pct") for ts in marker_sets])

print(f"\nmean useful_answer_pct over marker-only sets:")
print(f"  gemma:   {mean_a}")
print(f"  phi4:    {mean_b}")

print(f"\nTrain time / VRAM:")
print(f"  gemma:   {a['train_seconds']:.0f}s, {a['peak_vram_gb']:.1f} GB")
print(f"  phi4:    {b['train_seconds']:.0f}s, {b['peak_vram_gb']:.1f} GB")

if mean_a > mean_b:
    winner_key, winner = "gemma", a
elif mean_b > mean_a:
    winner_key, winner = "phi4", b
else:
    # Tie -> prefer smaller model (gemma).
    winner_key, winner = "gemma", a

verdict_path = Path(OUT_DIR) / "winner.json"
verdict_path.write_text(json.dumps({
    "winner": winner_key,
    "model_id": winner["model"],
    "metric": "mean_useful_answer_pct_marker_only",
    "scores": {"gemma": mean_a, "phi4": mean_b},
    "rationale": "Highest mean useful_answer_pct on marker-only test sets (no prose leak); tie-break smaller model.",
    "comparison": winner["comparison"],
}, indent=2))
print(f"\nWINNER: {winner_key}  ({winner['model']})")
print(f"Saved verdict -> {verdict_path}")
print(f"\nNext step: open 02_train_tier1.ipynb, set MODEL_NAME = '{winner['model']}', Run all.")

In [ ]:
# =====================================================================
# Cell 8 — Zip and download verdict + per-candidate eval traces.
# =====================================================================
import shutil
from pathlib import Path

artifact = Path(OUT_DIR) / "bakeoff_artifacts"
artifact.mkdir(exist_ok=True)
(artifact / "winner.json").write_bytes((Path(OUT_DIR) / "winner.json").read_bytes())
(artifact / "results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

# Copy each candidate's eval/ subtree (per-test-set summaries + comparison.json)
for cand in ("gemma", "phi4"):
    src = Path(OUT_DIR) / cand / "eval"
    if src.exists():
        shutil.copytree(src, artifact / cand / "eval", dirs_exist_ok=True)

zip_path = shutil.make_archive(str(Path(OUT_DIR) / "bakeoff_results"), "zip", artifact)
print(f"Zipped -> {zip_path}")

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print(f"Not in Colab. Files saved at: {artifact}")
